In [1]:

# ============================================================
# Step 1: Load Mushroom Dataset — Orange Comparison
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# Project directories
PROJECT_DIR = Path.cwd()
SPLIT_DIR = PROJECT_DIR / "outputs" / "data_split"
TRAINING_DIR = PROJECT_DIR / "outputs" / "training"

# Separate folder for Orange results
ORANGE_DIR = PROJECT_DIR / "reproducibility_orange"
ORANGE_DIR.mkdir(parents=True, exist_ok=True)

# Load AVISTA's saved data partitions
data = {
    "X_train": np.load(SPLIT_DIR / "X_train.npy"),
    "y_train": np.load(SPLIT_DIR / "y_train_encoded.npy"),

    "X_train_balanced": np.load(
        SPLIT_DIR / "X_train_balanced.npy"
    ),
    "y_train_balanced": np.load(
        SPLIT_DIR / "y_train_balanced_encoded.npy"
    ),

    "X_val": np.load(SPLIT_DIR / "X_val.npy"),
    "y_val": np.load(SPLIT_DIR / "y_val_encoded.npy"),

    "X_test": np.load(SPLIT_DIR / "X_test.npy"),
    "y_test": np.load(SPLIT_DIR / "y_test_encoded.npy"),
}

# Verify all partitions
for split in ["train", "val", "test"]:
    assert len(data[f"X_{split}"]) == len(data[f"y_{split}"])

assert len(data["X_train_balanced"]) == len(data["y_train_balanced"])

# Display shapes and class distributions
print("PROJECT:", PROJECT_DIR.name)
print("Orange results:", ORANGE_DIR)
print("=" * 65)

for split in ["train", "val", "test"]:
    X = data[f"X_{split}"]
    y = data[f"y_{split}"]

    classes, counts = np.unique(y, return_counts=True)

    print(f"\n{split.upper()}")
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Class distribution:", dict(zip(classes.tolist(), counts.tolist())))

print("\nBALANCED TRAINING")
print("X shape:", data["X_train_balanced"].shape)
print("y shape:", data["y_train_balanced"].shape)

print("\nData loaded successfully.")

PROJECT: national-poll2
Orange results: d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproducibility_orange

TRAIN
X shape: (499, 60)
y shape: (499,)
Class distribution: {0: 88, 1: 260, 2: 151}

VAL
X shape: (72, 60)
y shape: (72,)
Class distribution: {0: 15, 1: 38, 2: 19}

TEST
X shape: (143, 60)
y shape: (143,)
Class distribution: {0: 28, 1: 74, 2: 41}

BALANCED TRAINING
X shape: (541, 60)
y shape: (541,)

Data loaded successfully.


In [2]:
# %pip install orange3

In [3]:
import sys
import Orange
from Orange.data import Table, Domain, ContinuousVariable, DiscreteVariable

print("Python environment:", sys.executable)
print("Orange version:", Orange.__version__)
print("Orange imported successfully.")

Python environment: c:\Users\monzu\anaconda3\envs\mimi\python.exe
Orange version: 3.40.0
Orange imported successfully.


In [4]:

# ============================================================
# Step 2: Convert AVISTA Arrays to Orange Tables
# Binary and Multiclass Compatible
# ============================================================

import numpy as np
import Orange
from Orange.data import (
    Table, Domain, ContinuousVariable, DiscreteVariable
)

# Detect features and classes
n_features = data["X_train"].shape[1]

all_labels = np.concatenate([
    data["y_train"],
    data["y_val"],
    data["y_test"]
]).astype(int)

class_ids = np.unique(all_labels)
n_classes = len(class_ids)

# AVISTA LabelEncoder normally uses 0, 1, ..., K-1
assert np.array_equal(
    class_ids, np.arange(n_classes)
), "Target class indices are not contiguous."

# Create feature variables
features = [
    ContinuousVariable(f"feature_{i+1}")
    for i in range(n_features)
]

# Create multiclass-compatible target variable
# Names are placeholders; encoded indices are preserved.
target = DiscreteVariable(
    "target_class",
    values=[f"class_{i}" for i in range(n_classes)]
)

orange_domain = Domain(features, target)


def to_orange_table(X, y):
    return Table.from_numpy(
        orange_domain,
        X=np.asarray(X, dtype=np.float64),
        Y=np.asarray(y, dtype=np.float64)
    )


# Recreate all tables
orange_train = to_orange_table(
    data["X_train"], data["y_train"]
)

orange_train_balanced = to_orange_table(
    data["X_train_balanced"], data["y_train_balanced"]
)

orange_val = to_orange_table(
    data["X_val"], data["y_val"]
)

orange_test = to_orange_table(
    data["X_test"], data["y_test"]
)

# Verification
print("Dataset:", PROJECT_DIR.name)
print("Orange version:", Orange.__version__)
print("Features:", n_features)
print("Classes:", n_classes)
print("Class IDs:", class_ids)
print("Target categories:", target.values)

for name, table in {
    "Train": orange_train,
    "Balanced Train": orange_train_balanced,
    "Validation": orange_val,
    "Test": orange_test
}.items():
    assert np.all(
        (table.Y >= 0) & (table.Y < n_classes)
    )
    print(f"{name}: {len(table)} rows")

print("\nOrange tables verified successfully.")

Dataset: national-poll2
Orange version: 3.40.0
Features: 60
Classes: 3
Class IDs: [0 1 2]
Target categories: ('class_0', 'class_1', 'class_2')
Train: 499 rows
Balanced Train: 541 rows
Validation: 72 rows
Test: 143 rows

Orange tables verified successfully.


In [5]:

# ============================================================
# Step 3: Orange Logistic Regression
# AVISTA vs. Orange Python API
# Compatible with Mushroom, Obesity, and NPHA
# ============================================================

import json
import warnings
import joblib
import numpy as np
import pandas as pd
import Orange

from Orange.classification import LogisticRegressionLearner
from Orange.base import Model

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix
)

# ============================================================
# 1. LOAD AVISTA REFERENCE RESULTS
# ============================================================

with open(
    TRAINING_DIR / "training_results.json",
    "r",
    encoding="utf-8"
) as f:
    avista_results = json.load(f)

lr_reference = next(
    result
    for result in avista_results["results"]
    if "logistic regression" in result["model_name"].lower()
)

# Load AVISTA model configuration
lr_model_dir = TRAINING_DIR / "LogisticRegression"

with open(
    lr_model_dir / "model_config.json",
    "r",
    encoding="utf-8"
) as f:
    lr_config = json.load(f)

# Load the actual AVISTA fitted estimator to recover
# effective parameters, including class_weight.
# Only load trusted local AVISTA-generated model files.

with warnings.catch_warnings(record=True) as load_warnings:
    avista_lr_model = joblib.load(
        lr_model_dir / "trained_model.joblib"
    )

for warning_item in load_warnings:
    print("Model loading warning:", warning_item.message)

# ============================================================
# 2. VERIFY DATA AND TARGET CLASSES
# ============================================================

tables = {
    "Train": orange_train_balanced,
    "Validation": orange_val,
    "Test": orange_test
}

class_ids = np.unique(
    np.concatenate([
        np.asarray(data["y_train_balanced"], dtype=int),
        np.asarray(data["y_val"], dtype=int),
        np.asarray(data["y_test"], dtype=int)
    ])
)

n_classes = len(class_ids)

assert np.array_equal(
    class_ids, np.arange(n_classes)
), "Encoded class IDs must be 0, 1, ..., K-1."

for split_name, table in tables.items():

    assert len(table.domain.class_var.values) == n_classes, (
        f"{split_name}: Orange target domain has "
        f"{len(table.domain.class_var.values)} categories, "
        f"but dataset has {n_classes}. "
        "Rerun corrected Step 2."
    )

    assert table.X.shape[1] == data["X_train"].shape[1]

assert np.array_equal(
    orange_train_balanced.Y.astype(int),
    np.asarray(data["y_train_balanced"], dtype=int)
)

print("=" * 75)
print("ORANGE LOGISTIC REGRESSION")
print("=" * 75)

print("Dataset:", PROJECT_DIR.name)
print("Orange version:", Orange.__version__)
print("Features:", orange_train_balanced.X.shape[1])
print("Classes:", n_classes)
print("Class IDs:", class_ids.tolist())
print("Training samples:", len(orange_train_balanced))

# ============================================================
# 3. CONFIGURE MODEL USING ACTUAL AVISTA PARAMETERS
# ============================================================

saved_params = lr_config["parameters"].copy()
effective_params = vars(avista_lr_model)

# AVISTA may apply class weights during training even
# when model_config.json records class_weight=None.
# Use the effective fitted-estimator values.

class_weights = effective_params.get(
    "class_weight",
    saved_params.get("class_weight")
)

random_state = effective_params.get(
    "random_state",
    lr_config.get("random_state", None)
)

orange_lr_params = {
    "penalty": saved_params.get("penalty", "l2"),
    "C": saved_params.get("C", 1.0),
    "tol": saved_params.get("tol", 0.0001),
    "solver": saved_params.get("solver", "lbfgs"),
    "max_iter": saved_params.get("max_iter", 100),
    "fit_intercept": saved_params.get("fit_intercept", True),
    "class_weight": class_weights,
    "random_state": random_state
}

print("\nEffective class weights:", class_weights)
print("Effective random state:", random_state)
print("Model parameters:", orange_lr_params)

# ============================================================
# 4. TRAIN ORANGE LOGISTIC REGRESSION
# ============================================================

orange_lr_learner = LogisticRegressionLearner(
    **orange_lr_params,
    preprocessors=[]
)

orange_lr_model = orange_lr_learner(
    orange_train_balanced
)

print("\nTraining completed.")
print(
    "Underlying estimator:",
    type(orange_lr_model.skl_model)
)
print(
    "Iterations:",
    orange_lr_model.skl_model.n_iter_
)

# Check learned class labels
print(
    "Fitted classes:",
    orange_lr_model.skl_model.classes_
)

assert np.array_equal(
    orange_lr_model.skl_model.classes_,
    class_ids
), "Fitted estimator classes do not match encoded target IDs."

# ============================================================
# 5. MULTICLASS-COMPATIBLE EVALUATION FUNCTION
# ============================================================


def evaluate_orange_lr(model, table):

    actual = np.asarray(
        table.Y, dtype=int
    ).reshape(-1)

    predictions = np.asarray(
        model(table, Model.Value),
        dtype=int
    ).reshape(-1)

    probabilities = np.asarray(
        model(table, Model.Probs),
        dtype=float
    )

    # Verify probability output
    if probabilities.ndim != 2:
        raise ValueError(
            f"Expected 2D probability array, "
            f"got shape {probabilities.shape}"
        )

    if probabilities.shape[1] != n_classes:
        raise ValueError(
            f"Expected {n_classes} probability columns, "
            f"got {probabilities.shape[1]}"
        )

    metrics = {
        "accuracy": accuracy_score(
            actual, predictions
        ),

        "macro_f1": f1_score(
            actual,
            predictions,
            labels=class_ids,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            actual,
            predictions,
            labels=class_ids,
            average="weighted",
            zero_division=0
        ),

        "balanced_accuracy": balanced_accuracy_score(
            actual, predictions
        ),

        "macro_precision": precision_score(
            actual,
            predictions,
            labels=class_ids,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            actual,
            predictions,
            labels=class_ids,
            average="macro",
            zero_division=0
        )
    }

    # Binary or multiclass ROC-AUC
    try:

        if n_classes == 2:

            auc_value = roc_auc_score(
                actual,
                probabilities[:, 1]
            )

        else:

            auc_value = roc_auc_score(
                actual,
                probabilities,
                labels=class_ids,
                multi_class="ovr",
                average="macro"
            )

        metrics["roc_auc"] = float(auc_value)

    except ValueError:
        # ROC-AUC can be undefined if a split
        # does not contain all required classes.
        metrics["roc_auc"] = np.nan

    return metrics, predictions, probabilities


# ============================================================
# 6. COMPARE AVISTA AND ORANGE RESULTS
# ============================================================

comparison_rows = []
orange_lr_predictions = {}

for split, table in tables.items():

    orange_metrics, predictions, probabilities = (
        evaluate_orange_lr(
            orange_lr_model,
            table
        )
    )

    orange_lr_predictions[split] = predictions

    reference_metrics = lr_reference[
        f"{split.lower()}_metrics"
    ]

    for metric, orange_value in orange_metrics.items():

        avista_value = reference_metrics.get(
            metric, np.nan
        )

        if avista_value is None:
            avista_value = np.nan

        difference = (
            abs(orange_value - avista_value)
            if pd.notna(orange_value)
            and pd.notna(avista_value)
            else np.nan
        )

        comparison_rows.append({
            "Dataset": PROJECT_DIR.name,
            "Model": "Logistic Regression",
            "Split": split,
            "Metric": metric,
            "AVISTA": avista_value,
            "Orange": orange_value,
            "Absolute Difference": difference
        })

orange_lr_df = pd.DataFrame(comparison_rows)

for split in tables:

    print("\n" + "=" * 75)
    print(f"{split.upper()} COMPARISON")
    print("=" * 75)

    display(
        orange_lr_df[
            orange_lr_df["Split"] == split
        ][[
            "Metric",
            "AVISTA",
            "Orange",
            "Absolute Difference"
        ]].round(8)
    )


# ============================================================
# 7. TEST CONFUSION MATRIX — ALL CLASSES
# ============================================================

avista_cm = np.asarray(
    lr_reference["test_metrics"]["confusion_matrix"]
)

orange_cm = confusion_matrix(
    np.asarray(data["y_test"], dtype=int),
    orange_lr_predictions["Test"],
    labels=class_ids
)

print("\n" + "=" * 75)
print("TEST CONFUSION MATRICES")
print("=" * 75)

print("\nAVISTA confusion matrix:")
print(avista_cm)

print("\nOrange confusion matrix:")
print(orange_cm)

print(
    "\nConfusion matrices identical:",
    np.array_equal(avista_cm, orange_cm)
)

# Note: AVISTA may report decoded target labels.
# Confirm the class order before interpreting elementwise
# confusion-matrix agreement for multiclass datasets.

# ============================================================
# 8. SAVE RESULTS
# ============================================================

ORANGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    ORANGE_DIR / "LR_orange_reproducibility_results.csv"
)

orange_lr_df.to_csv(
    output_path,
    index=False
)

print("\n" + "=" * 75)
print("RESULTS SAVED")
print("=" * 75)

print(output_path)

print("\nOrange Logistic Regression experiment completed.")

Model loading warning: Trying to unpickle estimator LogisticRegression from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
ORANGE LOGISTIC REGRESSION
Dataset: national-poll2
Orange version: 3.40.0
Features: 60
Classes: 3
Class IDs: [0, 1, 2]
Training samples: 541

Effective class weights: {0: 1.3871794871794871, 1: 0.6935897435897436, 2: 1.1942604856512142}
Effective random state: None
Model parameters: {'penalty': 'l2', 'C': 1.0, 'tol': 0.0001, 'solver': 'lbfgs', 'max_iter': 100, 'fit_intercept': True, 'class_weight': {0: 1.3871794871794871, 1: 0.6935897435897436, 2: 1.1942604856512142}, 'random_state': None}

Training completed.
Underlying estimator: <class 'sklearn.linear_model._logistic.LogisticRegression'>
Iterations: [95]
Fitted classes: [0. 1. 2.]

TRAIN COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
0,accuracy,0.517560,0.517560,0.0
1,macro_f1,0.516819,0.516819,0.0
2,weighted_f1,0.519335,0.519335,0.0
3,balanced_accuracy,0.544974,0.544974,0.0
4,macro_precision,0.527317,0.527317,0.0
5,macro_recall,0.544974,0.544974,0.0
6,roc_auc,0.699686,0.699686,0.0



VALIDATION COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
7,accuracy,0.361111,0.361111,0.0
8,macro_f1,0.367735,0.367735,0.0
9,weighted_f1,0.349822,0.349822,0.0
10,balanced_accuracy,0.409942,0.409942,0.0
11,macro_precision,0.361270,0.361270,0.0
12,macro_recall,0.409942,0.409942,0.0
13,roc_auc,0.575776,0.575776,0.0



TEST COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
14,accuracy,0.398601,0.398601,0.0
15,macro_f1,0.376359,0.376359,0.0
16,weighted_f1,0.405804,0.405804,0.0
17,balanced_accuracy,0.392096,0.392096,0.0
18,macro_precision,0.388156,0.388156,0.0
19,macro_recall,0.392096,0.392096,0.0
20,roc_auc,0.600150,0.600150,0.0



TEST CONFUSION MATRICES

AVISTA confusion matrix:
[[ 8 10 10]
 [15 28 31]
 [10 10 21]]

Orange confusion matrix:
[[ 8 10 10]
 [15 28 31]
 [10 10 21]]

Confusion matrices identical: True

RESULTS SAVED
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproducibility_orange\LR_orange_reproducibility_results.csv

Orange Logistic Regression experiment completed.


In [6]:

# ============================================================
# Step 4: Orange Random Forest
# AVISTA vs. Orange Python API
# Compatible with Mushroom, Obesity, and NPHA
# ============================================================

import json
import inspect
import warnings
import joblib
import numpy as np
import pandas as pd
import Orange

from Orange.classification import RandomForestLearner
from Orange.base import Model

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. LOAD AVISTA REFERENCE AND CONFIGURATION
# ============================================================

rf_reference = next(
    result
    for result in avista_results["results"]
    if "random forest" in result["model_name"].lower()
)

rf_model_dir = TRAINING_DIR / "RandomForest"

with open(
    rf_model_dir / "model_config.json",
    "r",
    encoding="utf-8"
) as f:
    rf_config = json.load(f)

# Load AVISTA's actual fitted model to inspect
# effective class weights and other settings.
# Only load trusted local AVISTA-generated artifacts.

with warnings.catch_warnings(record=True) as load_warnings:
    avista_rf_model = joblib.load(
        rf_model_dir / "trained_model.joblib"
    )

for warning_item in load_warnings:
    print("Model loading warning:", warning_item.message)

# ============================================================
# 2. VERIFY DATA AND TARGET CLASSES
# ============================================================

tables = {
    "Train": orange_train_balanced,
    "Validation": orange_val,
    "Test": orange_test
}

rf_class_ids = np.unique(
    np.concatenate([
        np.asarray(data["y_train_balanced"], dtype=int),
        np.asarray(data["y_val"], dtype=int),
        np.asarray(data["y_test"], dtype=int)
    ])
)

n_classes = len(rf_class_ids)

assert np.array_equal(
    rf_class_ids,
    np.arange(n_classes)
), "Encoded target classes must be 0, 1, ..., K-1."

for split_name, table in tables.items():

    assert len(table.domain.class_var.values) == n_classes, (
        f"{split_name}: Orange target domain does not "
        f"contain all {n_classes} classes. "
        "Rerun corrected Step 2."
    )

    assert table.X.shape[1] == data["X_train"].shape[1]

assert np.array_equal(
    orange_train_balanced.Y.astype(int),
    np.asarray(data["y_train_balanced"], dtype=int)
)

print("=" * 75)
print("ORANGE RANDOM FOREST")
print("=" * 75)

print("Dataset:", PROJECT_DIR.name)
print("Orange version:", Orange.__version__)
print("Features:", orange_train_balanced.X.shape[1])
print("Classes:", n_classes)
print("Class IDs:", rf_class_ids.tolist())
print("Training samples:", len(orange_train_balanced))

# ============================================================
# 3. RECONSTRUCT EFFECTIVE MODEL PARAMETERS
# ============================================================

rf_params = rf_config["parameters"].copy()

# Access saved estimator attributes directly because
# AVISTA and the notebook may use different sklearn versions.

effective_state = vars(avista_rf_model)

# Use actual class weights rather than assuming every
# dataset uses the same balancing configuration.

rf_params["class_weight"] = effective_state.get(
    "class_weight",
    rf_params.get("class_weight")
)

rf_params["random_state"] = effective_state.get(
    "random_state",
    rf_config.get("random_state", 42)
)

# Orange exposes a subset of sklearn RF parameters.

supported = set(
    inspect.signature(RandomForestLearner).parameters
) - {"self", "preprocessors"}

orange_rf_params = {
    key: value
    for key, value in rf_params.items()
    if key in supported
}

unsupported_params = {
    key: value
    for key, value in rf_params.items()
    if key not in supported
}

print("\nEffective class weights:")
print(rf_params["class_weight"])

print("\nEffective random state:")
print(rf_params["random_state"])

print("\nOrange-supported parameters:")
print(orange_rf_params)

print("\nParameters not directly exposed by Orange:")
print(unsupported_params if unsupported_params else "None")

# ============================================================
# 4. TRAIN ORANGE RANDOM FOREST
# ============================================================

orange_rf_learner = RandomForestLearner(
    **orange_rf_params,
    preprocessors=[]
)

orange_rf_model = orange_rf_learner(
    orange_train_balanced
)

rf_estimator = orange_rf_model.skl_model

print("\nTraining completed.")

print("Underlying estimator:", type(rf_estimator))
print("Trees trained:", len(rf_estimator.estimators_))
print("Fitted classes:", rf_estimator.classes_)

assert np.array_equal(
    rf_estimator.classes_,
    rf_class_ids
), "Fitted estimator classes do not match target IDs."

# Verify that unsupported parameters match the
# effective defaults of Orange's sklearn estimator.

orange_effective = rf_estimator.get_params(deep=False)

print("\nUNEXPOSED PARAMETER VERIFICATION")
print("=" * 75)

parameter_mismatches = {}

for key, avista_value in unsupported_params.items():

    if key not in orange_effective:
        print(f"{key}: unavailable in Orange estimator")
        continue

    orange_value = orange_effective[key]

    matched = (repr(avista_value) == repr(orange_value))

    print(
        f"{key}: "
        f"AVISTA={avista_value!r}, "
        f"Orange={orange_value!r}, "
        f"Match={matched}"
    )

    if not matched:
        parameter_mismatches[key] = {
            "AVISTA": repr(avista_value),
            "Orange": repr(orange_value)
        }

if parameter_mismatches:
    print(
        "\nWARNING: Some effective parameters differ. "
        "Interpret metric agreement as performance "
        "consistency, not exact configuration replication."
    )
else:
    print("\nAll checked parameters match.")

# ============================================================
# 5. MULTICLASS-COMPATIBLE EVALUATION
# ============================================================


def evaluate_orange_rf(model, table):

    actual = np.asarray(
        table.Y,
        dtype=int
    ).reshape(-1)

    predictions = np.asarray(
        model(table, Model.Value),
        dtype=int
    ).reshape(-1)

    probabilities = np.asarray(
        model(table, Model.Probs),
        dtype=float
    )

    if probabilities.ndim != 2:
        raise ValueError(
            f"Expected 2D probabilities, "
            f"got {probabilities.shape}"
        )

    if probabilities.shape[1] != n_classes:
        raise ValueError(
            f"Expected {n_classes} probability columns, "
            f"got {probabilities.shape[1]}"
        )

    metrics = {
        "accuracy": accuracy_score(
            actual, predictions
        ),

        "macro_f1": f1_score(
            actual,
            predictions,
            labels=rf_class_ids,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            actual,
            predictions,
            labels=rf_class_ids,
            average="weighted",
            zero_division=0
        ),

        "balanced_accuracy": balanced_accuracy_score(
            actual,
            predictions
        ),

        "macro_precision": precision_score(
            actual,
            predictions,
            labels=rf_class_ids,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            actual,
            predictions,
            labels=rf_class_ids,
            average="macro",
            zero_division=0
        )
    }

    # Binary or multiclass ROC-AUC

    try:

        if n_classes == 2:

            auc_value = roc_auc_score(
                actual,
                probabilities[:, 1]
            )

        else:

            auc_value = roc_auc_score(
                actual,
                probabilities,
                labels=rf_class_ids,
                multi_class="ovr",
                average="macro"
            )

        metrics["roc_auc"] = float(auc_value)

    except ValueError:
        metrics["roc_auc"] = np.nan

    return metrics, predictions, probabilities


# ============================================================
# 6. TRAIN / VALIDATION / TEST COMPARISON
# ============================================================

comparison_rows = []
orange_rf_predictions = {}

for split, table in tables.items():

    orange_metrics, predictions, probabilities = (
        evaluate_orange_rf(
            orange_rf_model,
            table
        )
    )

    orange_rf_predictions[split] = predictions

    reference_metrics = rf_reference[
        f"{split.lower()}_metrics"
    ]

    for metric, orange_value in orange_metrics.items():

        avista_value = reference_metrics.get(
            metric, np.nan
        )

        if avista_value is None:
            avista_value = np.nan

        difference = (
            abs(orange_value - avista_value)
            if pd.notna(orange_value)
            and pd.notna(avista_value)
            else np.nan
        )

        comparison_rows.append({
            "Dataset": PROJECT_DIR.name,
            "Model": "Random Forest",
            "Split": split,
            "Metric": metric,
            "AVISTA": avista_value,
            "Orange": orange_value,
            "Absolute Difference": difference
        })

orange_rf_df = pd.DataFrame(comparison_rows)

for split in tables:

    print("\n" + "=" * 75)
    print(f"{split.upper()} COMPARISON")
    print("=" * 75)

    display(
        orange_rf_df[
            orange_rf_df["Split"] == split
        ][[
            "Metric",
            "AVISTA",
            "Orange",
            "Absolute Difference"
        ]].round(8)
    )


# ============================================================
# 7. TEST CONFUSION MATRIX — ALL CLASSES
# ============================================================

avista_cm = np.asarray(
    rf_reference["test_metrics"]["confusion_matrix"]
)

orange_cm = confusion_matrix(
    np.asarray(data["y_test"], dtype=int),
    orange_rf_predictions["Test"],
    labels=rf_class_ids
)

print("\n" + "=" * 75)
print("TEST CONFUSION MATRICES")
print("=" * 75)

print("\nAVISTA confusion matrix:")
print(avista_cm)

print("\nOrange confusion matrix:")
print(orange_cm)

print(
    "\nConfusion matrices numerically identical:",
    np.array_equal(avista_cm, orange_cm)
)

print("\nOrange classification report:")
print(
    classification_report(
        np.asarray(data["y_test"], dtype=int),
        orange_rf_predictions["Test"],
        labels=rf_class_ids,
        zero_division=0
    )
)

# AVISTA may report decoded class labels.
# Confirm class ordering before interpreting an
# elementwise confusion-matrix comparison.

# ============================================================
# 8. SAVE RESULTS
# ============================================================

ORANGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

rf_output_path = (
    ORANGE_DIR / "RF_orange_reproducibility_results.csv"
)

orange_rf_df.to_csv(
    rf_output_path,
    index=False
)

print("\n" + "=" * 75)
print("RESULTS SAVED")
print("=" * 75)

print(rf_output_path)

print("\nOrange Random Forest experiment completed.")

Model loading warning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Model loading warning: Trying to unpickle estimator RandomForestClassifier from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
ORANGE RANDOM FOREST
Dataset: national-poll2
Orange version: 3.40.0
Features: 60
Classes: 3
Class IDs: [0, 1, 2]
Training samples: 541

Effective class weights:
{0: 1.3871794871794871, 1: 0.6935897435897436, 2: 1.1942604856512142}

Effective random state:
None

Orange-supported parameters:
{'n_estimators': 100, 'criterion': 'gini', 'max_depth': None, 'min_sample

,Metric,AVISTA,Orange,Absolute Difference
0,accuracy,0.950092,0.951941,0.001848
1,macro_f1,0.946386,0.947985,0.001599
2,weighted_f1,0.950580,0.952064,0.001484
3,balanced_accuracy,0.952284,0.950289,0.001995
4,macro_precision,0.942785,0.945862,0.003077
5,macro_recall,0.952284,0.950289,0.001995
6,roc_auc,0.995412,0.994555,0.000858



VALIDATION COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
7,accuracy,0.402778,0.416667,0.013889
8,macro_f1,0.337217,0.264665,0.072553
9,weighted_f1,0.400717,0.361995,0.038722
10,balanced_accuracy,0.342690,0.298246,0.044444
11,macro_precision,0.341595,0.240741,0.100855
12,macro_recall,0.342690,0.298246,0.044444
13,roc_auc,0.595291,0.564462,0.030829



TEST COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
14,accuracy,0.433566,0.447552,0.013986
15,macro_f1,0.361977,0.313535,0.048442
16,weighted_f1,0.424204,0.399006,0.025198
17,balanced_accuracy,0.359787,0.332093,0.027694
18,macro_precision,0.381403,0.364996,0.016407
19,macro_recall,0.359787,0.332093,0.027694
20,roc_auc,0.545571,0.518760,0.026811



TEST CONFUSION MATRICES

AVISTA confusion matrix:
[[ 5 13 10]
 [ 4 45 25]
 [ 6 23 12]]

Orange confusion matrix:
[[ 2 19  7]
 [ 2 54 18]
 [ 2 31  8]]

Confusion matrices numerically identical: False

Orange classification report:
              precision    recall  f1-score   support

           0       0.33      0.07      0.12        28
           1       0.52      0.73      0.61        74
           2       0.24      0.20      0.22        41

    accuracy                           0.45       143
   macro avg       0.36      0.33      0.31       143
weighted avg       0.40      0.45      0.40       143


RESULTS SAVED
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproducibility_orange\RF_orange_reproducibility_results.csv

Orange Random Forest experiment completed.


In [7]:

# ============================================================
# Step 5: Orange XGBoost
# AVISTA vs. Orange Python API
# Compatible with Mushroom, Obesity, and NPHA
# ============================================================

import json
import inspect
import numpy as np
import pandas as pd
import Orange
import xgboost as xgb

from Orange.classification.xgb import XGBClassifier as OrangeXGBClassifier
from Orange.base import Model

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. LOAD AVISTA REFERENCE AND CONFIGURATION
# ============================================================

with open(
    TRAINING_DIR / "training_results.json",
    "r", encoding="utf-8"
) as f:
    avista_results = json.load(f)

xgb_reference = next(
    result for result in avista_results["results"]
    if "xgboost" in result["model_name"].lower()
)

with open(
    TRAINING_DIR / "XGBoost" / "model_config.json",
    "r", encoding="utf-8"
) as f:
    xgb_config = json.load(f)

xgb_params = xgb_config["parameters"].copy()

if (
    xgb_params.get("random_state") is None
    and xgb_config.get("random_state") is not None
):
    xgb_params["random_state"] = xgb_config["random_state"]

# ============================================================
# 2. VERIFY MULTICLASS DATA
# ============================================================

tables = {
    "Train": orange_train_balanced,
    "Validation": orange_val,
    "Test": orange_test
}

xgb_class_ids = np.unique(
    np.concatenate([
        np.asarray(data["y_train_balanced"], dtype=int),
        np.asarray(data["y_val"], dtype=int),
        np.asarray(data["y_test"], dtype=int)
    ])
)

n_classes = len(xgb_class_ids)

assert np.array_equal(
    xgb_class_ids, np.arange(n_classes)
), "Encoded classes must be 0, 1, ..., K-1."

assert np.array_equal(
    np.unique(data["y_train_balanced"]),
    xgb_class_ids
), "Not all target classes are present in the training data."

for split, table in tables.items():

    assert len(table.domain.class_var.values) == n_classes, (
        f"{split}: Orange target domain does not match "
        f"the {n_classes} target classes. Rerun Step 2."
    )

    assert table.X.shape[1] == data["X_train"].shape[1]

print("=" * 75)
print("ORANGE XGBOOST")
print("=" * 75)

print("Dataset:", PROJECT_DIR.name)
print("Orange version:", Orange.__version__)
print("XGBoost version:", xgb.__version__)
print("Features:", orange_train_balanced.X.shape[1])
print("Classes:", n_classes)
print("Class IDs:", xgb_class_ids.tolist())
print("Training samples:", len(orange_train_balanced))

# ============================================================
# 3. RECONSTRUCT ORANGE-SUPPORTED PARAMETERS
# ============================================================

supported = set(
    inspect.signature(OrangeXGBClassifier).parameters
) - {"self", "preprocessors"}

orange_xgb_params = {
    key: value
    for key, value in xgb_params.items()
    if key in supported
}

unsupported_params = {
    key: value
    for key, value in xgb_params.items()
    if key not in supported
}

print("\nAVISTA XGBoost parameters:")
print(xgb_params)

print("\nParameters not directly exposed by Orange:")
print(unsupported_params if unsupported_params else "None")

# Do not silently omit settings that may change fitting.
# eval_metric only controls evaluation, and
# enable_categorical=False is the expected numeric-data mode.

allowed_omissions = {
    "eval_metric",
    "enable_categorical"
}

unexpected_omissions = {
    key: value
    for key, value in unsupported_params.items()
    if key not in allowed_omissions
}

if unexpected_omissions:
    raise ValueError(
        "Orange does not expose some potentially "
        "important AVISTA parameters: "
        f"{unexpected_omissions}"
    )

if unsupported_params.get("enable_categorical", False):
    raise ValueError(
        "AVISTA used native categorical XGBoost, "
        "which this comparison does not reproduce."
    )

# XGBoost handles multiclass objectives internally.
# Keep the saved AVISTA objective rather than guessing
# a replacement based on the number of classes.

print("\nObjective from AVISTA config:",
      xgb_params.get("objective"))

# ============================================================
# 4. TRAIN ORANGE XGBOOST
# ============================================================

orange_xgb_learner = OrangeXGBClassifier(
    **orange_xgb_params,
    preprocessors=[]
)

orange_xgb_model = orange_xgb_learner(
    orange_train_balanced
)

xgb_estimator = orange_xgb_model.skl_model

print("\nTraining completed.")
print("Underlying estimator:", type(xgb_estimator))
print("Estimators:", xgb_estimator.n_estimators)
print("Fitted classes:", xgb_estimator.classes_)

assert np.array_equal(
    np.asarray(xgb_estimator.classes_, dtype=int),
    xgb_class_ids
), "Fitted class ordering differs from the encoded labels."

# XGBoost can select multi:softprob internally
# even when its constructor specified binary:logistic.

booster_config = json.loads(
    xgb_estimator.get_booster().save_config()
)

effective_objective = (
    booster_config["learner"]["objective"]["name"]
)

print("Effective trained objective:", effective_objective)

if n_classes > 2 and effective_objective != "multi:softprob":
    raise ValueError(
        "Unexpected trained multiclass objective: "
        f"{effective_objective}"
    )

# ============================================================
# 5. BINARY / MULTICLASS EVALUATION
# ============================================================


def evaluate_orange_xgb(model, table):

    actual = np.asarray(
        table.Y, dtype=int
    ).reshape(-1)

    predictions = np.asarray(
        model(table, Model.Value),
        dtype=int
    ).reshape(-1)

    probabilities = np.asarray(
        model(table, Model.Probs),
        dtype=float
    )

    if probabilities.ndim != 2:
        raise ValueError(
            f"Expected 2D probabilities, got "
            f"{probabilities.shape}"
        )

    if probabilities.shape[1] != n_classes:
        raise ValueError(
            f"Expected {n_classes} probability columns, "
            f"got {probabilities.shape[1]}"
        )

    metrics = {
        "accuracy": accuracy_score(
            actual, predictions
        ),

        "macro_f1": f1_score(
            actual, predictions,
            labels=xgb_class_ids,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            actual, predictions,
            labels=xgb_class_ids,
            average="weighted",
            zero_division=0
        ),

        "balanced_accuracy": balanced_accuracy_score(
            actual, predictions
        ),

        "macro_precision": precision_score(
            actual, predictions,
            labels=xgb_class_ids,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            actual, predictions,
            labels=xgb_class_ids,
            average="macro",
            zero_division=0
        )
    }

    # ROC-AUC: binary or macro one-vs-rest multiclass

    try:
        if n_classes == 2:

            auc_value = roc_auc_score(
                actual,
                probabilities[:, 1]
            )

        else:

            auc_value = roc_auc_score(
                actual,
                probabilities,
                labels=xgb_class_ids,
                multi_class="ovr",
                average="macro"
            )

        metrics["roc_auc"] = float(auc_value)

    except ValueError:
        metrics["roc_auc"] = np.nan

    return metrics, predictions, probabilities


# ============================================================
# 6. TRAIN / VALIDATION / TEST COMPARISON
# ============================================================

comparison_rows = []
orange_xgb_predictions = {}

for split, table in tables.items():

    orange_metrics, predictions, probabilities = (
        evaluate_orange_xgb(
            orange_xgb_model,
            table
        )
    )

    orange_xgb_predictions[split] = predictions

    reference_metrics = xgb_reference[
        f"{split.lower()}_metrics"
    ]

    for metric, orange_value in orange_metrics.items():

        avista_value = reference_metrics.get(
            metric, np.nan
        )

        if avista_value is None:
            avista_value = np.nan

        difference = (
            abs(orange_value - avista_value)
            if pd.notna(orange_value)
            and pd.notna(avista_value)
            else np.nan
        )

        comparison_rows.append({
            "Dataset": PROJECT_DIR.name,
            "Model": "XGBoost",
            "Split": split,
            "Metric": metric,
            "AVISTA": avista_value,
            "Orange": orange_value,
            "Absolute Difference": difference
        })

orange_xgb_df = pd.DataFrame(comparison_rows)

for split in tables:

    print("\n" + "=" * 75)
    print(f"{split.upper()} COMPARISON")
    print("=" * 75)

    display(
        orange_xgb_df[
            orange_xgb_df["Split"] == split
        ][[
            "Metric",
            "AVISTA",
            "Orange",
            "Absolute Difference"
        ]].round(8)
    )


# ============================================================
# 7. TEST CONFUSION MATRIX — ALL CLASSES
# ============================================================

avista_cm = np.asarray(
    xgb_reference["test_metrics"]["confusion_matrix"]
)

orange_cm = confusion_matrix(
    np.asarray(data["y_test"], dtype=int),
    orange_xgb_predictions["Test"],
    labels=xgb_class_ids
)

print("\n" + "=" * 75)
print("TEST CONFUSION MATRICES")
print("=" * 75)

print("\nAVISTA:")
print(avista_cm)

print("\nOrange:")
print(orange_cm)

print(
    "\nConfusion matrices numerically identical:",
    np.array_equal(avista_cm, orange_cm)
)

print("\nOrange classification report:")
print(
    classification_report(
        np.asarray(data["y_test"], dtype=int),
        orange_xgb_predictions["Test"],
        labels=xgb_class_ids,
        zero_division=0
    )
)

# AVISTA may report decoded target labels.
# Verify their order before interpreting confusion
# matrix entries for multiclass datasets.

# ============================================================
# 8. SAVE RESULTS
# ============================================================

ORANGE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

xgb_output_path = (
    ORANGE_DIR / "XGB_orange_reproducibility_results.csv"
)

orange_xgb_df.to_csv(
    xgb_output_path,
    index=False
)

config_output_path = (
    ORANGE_DIR / "XGB_orange_configuration_audit.json"
)

with open(
    config_output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "dataset": PROJECT_DIR.name,
            "orange_version": Orange.__version__,
            "xgboost_version": xgb.__version__,
            "n_classes": n_classes,
            "avista_parameters": xgb_params,
            "orange_parameters": orange_xgb_params,
            "unsupported_parameters": unsupported_params,
            "effective_trained_objective": effective_objective
        },
        f,
        indent=2,
        default=str
    )

print("\n" + "=" * 75)
print("RESULTS SAVED")
print("=" * 75)

print(xgb_output_path)
print(config_output_path)

print("\nOrange XGBoost experiment completed.")

ORANGE XGBOOST
Dataset: national-poll2
Orange version: 3.40.0
XGBoost version: 3.0.2
Features: 60
Classes: 3
Class IDs: [0, 1, 2]
Training samples: 541

AVISTA XGBoost parameters:
{'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'objective': 'multi:softprob', 'eval_metric': 'mlogloss', 'tree_method': 'auto', 'booster': 'gbtree', 'gamma': 0.0, 'min_child_weight': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'scale_pos_weight': 1.0, 'random_state': 42, 'n_jobs': None, 'verbosity': 0, 'enable_categorical': False}

Parameters not directly exposed by Orange:
{'eval_metric': 'mlogloss', 'enable_categorical': False}

Objective from AVISTA config: multi:softprob

Training completed.
Underlying estimator: <class 'xgboost.sklearn.XGBClassifier'>
Estimators: 300
Fitted classes: [0 1 2]
Effective trained objective: multi:softprob

TRAIN COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
0,accuracy,0.905730,0.903882,0.001848
1,macro_f1,0.899450,0.898242,0.001208
2,weighted_f1,0.904638,0.902897,0.001741
3,balanced_accuracy,0.883698,0.882416,0.001282
4,macro_precision,0.922999,0.921857,0.001142
5,macro_recall,0.883698,0.882416,0.001282
6,roc_auc,0.983431,0.983302,0.000128



VALIDATION COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
7,accuracy,0.416667,0.388889,0.027778
8,macro_f1,0.281661,0.268111,0.013550
9,weighted_f1,0.374642,0.355065,0.019577
10,balanced_accuracy,0.307018,0.289474,0.017544
11,macro_precision,0.262661,0.250840,0.011821
12,macro_recall,0.307018,0.289474,0.017544
13,roc_auc,0.539835,0.550907,0.011073



TEST COMPARISON


,Metric,AVISTA,Orange,Absolute Difference
14,accuracy,0.475524,0.475524,0.000000
15,macro_f1,0.382068,0.390165,0.008097
16,weighted_f1,0.450228,0.454331,0.004103
17,balanced_accuracy,0.383040,0.386815,0.003775
18,macro_precision,0.452224,0.448718,0.003507
19,macro_recall,0.383040,0.386815,0.003775
20,roc_auc,0.558880,0.558966,0.000086



TEST CONFUSION MATRICES

AVISTA:
[[ 4 15  9]
 [ 2 51 21]
 [ 2 26 13]]

Orange:
[[ 5 13 10]
 [ 3 51 20]
 [ 2 27 12]]

Confusion matrices numerically identical: False

Orange classification report:
              precision    recall  f1-score   support

           0       0.50      0.18      0.26        28
           1       0.56      0.69      0.62        74
           2       0.29      0.29      0.29        41

    accuracy                           0.48       143
   macro avg       0.45      0.39      0.39       143
weighted avg       0.47      0.48      0.45       143


RESULTS SAVED
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproducibility_orange\XGB_orange_reproducibility_results.csv
d:\TXST\OneDrive - Texas State University\Hackathon_Monzurul\Papers\AVISTA\examples\national-poll2\reproducibility_orange\XGB_orange_configuration_audit.json

Orange XGBoost experiment completed.


In [8]:

# ============================================================
# Step 6: Orange RF and XGBoost — Final verification
# ============================================================

for name, df in [
    ("Random Forest", orange_rf_df),
    ("XGBoost", orange_xgb_df)
]:
    print(f"\n{'=' * 70}")
    print(f"{name.upper()} — TEST RESULTS")
    print("=" * 70)

    test = df[
        (df["Split"] == "Test") &
        (df["Metric"].isin(["accuracy", "macro_f1", "roc_auc"]))
    ]

    display(test[
        ["Metric", "AVISTA", "Orange", "Absolute Difference"]
    ].round(10))

    print(
        "Maximum absolute difference across all reported metrics:",
        df["Absolute Difference"].max()
    )

print("\nXGBOOST EFFECTIVE PARAMETERS")
print("=" * 70)

xgb_effective = orange_xgb_model.skl_model.get_params()

for key in ["eval_metric", "enable_categorical",
            "n_estimators", "random_state"]:
    print(f"{key}: {xgb_effective.get(key)}")

print("\nRANDOM FOREST EFFECTIVE PARAMETERS")
print("=" * 70)

rf_effective = orange_rf_model.skl_model.get_params()

for key in [
    "min_impurity_decrease", "warm_start",
    "ccp_alpha", "max_samples", "monotonic_cst",
    "class_weight", "random_state"
]:
    print(f"{key}: {rf_effective.get(key)}")

print("\nSAVED FILES")
print("=" * 70)

for filename in [
    "LR_orange_reproducibility_results.csv",
    "RF_orange_reproducibility_results.csv",
    "XGB_orange_reproducibility_results.csv"
]:
    path = ORANGE_DIR / filename
    print(f"{filename}: {'Found' if path.exists() else 'Missing'}")


RANDOM FOREST — TEST RESULTS


,Metric,AVISTA,Orange,Absolute Difference
14,accuracy,0.433566,0.447552,0.013986
15,macro_f1,0.361977,0.313535,0.048442
20,roc_auc,0.545571,0.518760,0.026811


Maximum absolute difference across all reported metrics: 0.10085470085470086

XGBOOST — TEST RESULTS


,Metric,AVISTA,Orange,Absolute Difference
14,accuracy,0.475524,0.475524,0.000000
15,macro_f1,0.382068,0.390165,0.008097
20,roc_auc,0.558880,0.558966,0.000086


Maximum absolute difference across all reported metrics: 0.02777777777777779

XGBOOST EFFECTIVE PARAMETERS
eval_metric: None
enable_categorical: False
n_estimators: 300
random_state: 42

RANDOM FOREST EFFECTIVE PARAMETERS
min_impurity_decrease: 0.0
warm_start: False
ccp_alpha: 0.0
max_samples: None
monotonic_cst: None
class_weight: {0: 1.3871794871794871, 1: 0.6935897435897436, 2: 1.1942604856512142}
random_state: None

SAVED FILES
LR_orange_reproducibility_results.csv: Found
RF_orange_reproducibility_results.csv: Found
XGB_orange_reproducibility_results.csv: Found


In [9]:
# ============================================================
# Step 7: Consolidate Orange Comparison Results — Mushroom
# ============================================================

import pandas as pd

all_orange_results = pd.concat(
    [
        orange_lr_df,
        orange_rf_df,
        orange_xgb_df
    ],
    ignore_index=True
)

# Consolidated full results
all_orange_results.to_csv(
    ORANGE_DIR / "Orange_all_models_comparison.csv",
    index=False
)

# Publication-ready test summary
test_summary = (
    all_orange_results[
        (all_orange_results["Split"] == "Test") &
        (all_orange_results["Metric"].isin(
            ["accuracy", "macro_f1", "roc_auc"]
        ))
    ]
    .pivot_table(
        index=["Dataset", "Model"],
        columns="Metric",
        values=["AVISTA", "Orange"],
        aggfunc="first"
    )
)

test_summary.columns = [
    f"{platform}_{metric}"
    for platform, metric in test_summary.columns
]

test_summary = test_summary.reset_index()

display(test_summary.round(6))

test_summary.to_csv(
    ORANGE_DIR / "Orange_test_summary.csv",
    index=False
)

print("Consolidated Orange results saved.")

,Dataset,Model,AVISTA_accuracy,AVISTA_macro_f1,AVISTA_roc_auc,Orange_accuracy,Orange_macro_f1,Orange_roc_auc
0,national-poll2,Logistic Regression,0.398601,0.376359,0.600150,0.398601,0.376359,0.600150
1,national-poll2,Random Forest,0.433566,0.361977,0.545571,0.447552,0.313535,0.518760
2,national-poll2,XGBoost,0.475524,0.382068,0.558880,0.475524,0.390165,0.558966


Consolidated Orange results saved.
